# Open Financial Analyst — end-to-end notebook
Kaggle par chalane ke liye. Order mein top se bottom tak run karo.

## 1. Installs

In [ ]:
!pip install -q langchain-google-genai langchain-core langchain langgraph
!pip install -q sec-edgar-downloader beautifulsoup4 lxml
!pip install -q sentence-transformers faiss-cpu rank_bm25

## 2. LLM setup (Gemma 4 31B via Gemini API)

In [ ]:
from kaggle_secrets import UserSecretsClient
from langchain_google_genai import ChatGoogleGenerativeAI

secrets = UserSecretsClient()
GEMINI_KEY = secrets.get_secret("GEMINI_API_KEY")  # Kaggle Secrets mein label se
SEC_EMAIL = secrets.get_secret("SEC_EDGAR_EMAIL")   # SEC EDGAR fair-access policy ke liye

llm = ChatGoogleGenerativeAI(model="gemma-4-31b-it", google_api_key=GEMINI_KEY)
# Gemma 4 31B: open-weight (Apache 2.0), free tier ~15 RPM / 1M tokens per day, $0 cost

In [ ]:
def extract_text(response):
    """
    Gemma kabhi plain string deta hai, kabhi list-of-dicts (thinking + text blocks).
    Sirf 'text' type chahiye, 'thinking' wala internal reasoning nahi.
    """
    if isinstance(response.content, str):
        return response.content
    elif isinstance(response.content, list):
        return "".join(
            part.get("text", "") for part in response.content
            if isinstance(part, dict) and part.get("type") == "text"
        )
    return str(response.content)

In [ ]:
import time

# Phase 5 (cost/latency tracking) ke liye — har LLM call ka time aur count log hota hai
_llm_stats = {"total_calls": 0, "total_time": 0.0, "call_log": []}

def reset_llm_stats():
    global _llm_stats
    _llm_stats = {"total_calls": 0, "total_time": 0.0, "call_log": []}

def get_llm_stats():
    return dict(_llm_stats)


def invoke_with_retry(llm, prompt, max_retries=3, min_gap=2):
    """
    429 (rate limit) errors ke liye exponential backoff.
    min_gap: har call se pehle chhota pacing gap, RPM limit se bachne ke liye.
    """
    time.sleep(min_gap)
    for attempt in range(max_retries):
        start = time.time()
        try:
            response = llm.invoke(prompt)
            elapsed = time.time() - start
            _llm_stats["total_calls"] += 1
            _llm_stats["total_time"] += elapsed
            _llm_stats["call_log"].append(elapsed)
            return response
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) and attempt < max_retries - 1:
                wait_time = 2 ** attempt * 10  # 10s, 20s, 40s
                print(f"Rate limited, retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise
    return None

## 3. Data ingestion — SEC EDGAR

In [ ]:
from sec_edgar_downloader import Downloader
import os, re

dl = Downloader("OpenFinancialAnalyst", SEC_EMAIL)

# 6 companies, alag-alag sectors se — taaki filing table formats bhi vary karein
# aur system generalize karta hai ye prove ho
COMPANIES = {
    "NVIDIA": "NVDA",              # semiconductors
    "AMD": "AMD",                  # semiconductors
    "Apple": "AAPL",               # consumer tech
    "Walmart": "WMT",              # retail
    "Johnson & Johnson": "JNJ",    # healthcare
    "JPMorgan Chase": "JPM",       # banking/finance
}

for name, ticker in COMPANIES.items():
    print(f"Downloading {name} ({ticker})...")
    dl.get("10-K", ticker, limit=2)
    dl.get("10-Q", ticker, limit=4)

In [ ]:
def get_filing_period(filepath):
    """
    Har SEC filing ke header mein 'CONFORMED PERIOD OF REPORT' hota hai.
    Isse hum manually guess nahi karte, seedha file se actual date nikalte hain.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        header = f.read(3000)
    match = re.search(r"CONFORMED PERIOD OF REPORT:\s*(\d{8})", header)
    if match:
        date_str = match.group(1)  # YYYYMMDD
        return f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"
    return "unknown"


def build_filing_paths(ticker, folder="sec-edgar-filings"):
    """
    Ek ticker ke saare downloaded filings dhoondh ke, unka actual period
    (header se) nikaal ke, ek clean naam wala dict banata hai.
    """
    paths = {}
    for doc_type in ["10-K", "10-Q"]:
        base = f"{folder}/{ticker}/{doc_type}"
        if not os.path.exists(base):
            continue
        for accession_folder in os.listdir(base):
            filepath = os.path.join(base, accession_folder, "full-submission.txt")
            if os.path.exists(filepath):
                period = get_filing_period(filepath)
                key = f"{doc_type}_{period}"
                paths[key] = filepath
    return paths


filing_paths_by_company = {}
for name, ticker in COMPANIES.items():
    paths = build_filing_paths(ticker)
    filing_paths_by_company[name] = paths
    print(f"{name} ({ticker}): {len(paths)} filings found")

In [ ]:
from bs4 import BeautifulSoup

def extract_primary_document(filepath, doc_type="10-K"):
    # raw SEC file ek container hai jisme sainkdo documents bunde hote hain
    # yahan sirf primary filing document nikalna hai
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    documents = re.findall(r"<DOCUMENT>(.*?)</DOCUMENT>", content, re.DOTALL)
    for doc in documents:
        type_match = re.search(r"<TYPE>(.*?)\n", doc)
        if type_match and type_match.group(1).strip() == doc_type:
            text_match = re.search(r"<TEXT>(.*)", doc, re.DOTALL)
            if text_match:
                return text_match.group(1)
    return None


def parse_sec_filing(filepath, doc_type):
    raw_html = extract_primary_document(filepath, doc_type)
    if raw_html is None:
        return None, []

    html_content = re.sub(r"</?XBRL>", "", raw_html)
    soup = BeautifulSoup(html_content, "lxml")

    # hidden inline-XBRL metadata hatao
    for hidden in soup.find_all("ix:header"):
        hidden.decompose()
    for hidden in soup.find_all(style=re.compile(r"display\s*:\s*none")):
        hidden.decompose()

    # tables alag nikalo, structured rows ke roop mein
    tables = []
    for table in soup.find_all("table"):
        rows = []
        for tr in table.find_all("tr"):
            cells_ = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
            if any(cells_):
                rows.append(cells_)
        if rows:
            tables.append(rows)
        table.decompose()

    plain_text = soup.get_text(separator="\n", strip=True)
    return plain_text, tables

In [ ]:
def parse_all(filing_paths):
    parsed = {}
    for name, path in filing_paths.items():
        doc_type = "10-K" if "10-K" in name else "10-Q"
        text, tables = parse_sec_filing(path, doc_type=doc_type)
        parsed[name] = {"text": text, "tables": tables}
        print(f"{name}: text={len(text)} chars, tables={len(tables)}")
    return parsed


parsed_by_company = {}
for name, paths in filing_paths_by_company.items():
    print(f"--- Parsing {name} ---")
    parsed_by_company[name] = parse_all(paths)

## 4. Contextual chunking

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=150):
    # word-based chunking, overlap taaki chunk boundary pe sentence na kate
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks


def build_contextual_chunks(parsed, company):
    all_chunks = []
    for doc_name, data in parsed.items():
        doc_type, period = doc_name.split("_", 1)
        raw_chunks = chunk_text(data["text"])
        for i, chunk in enumerate(raw_chunks):
            # context prefix — company/doctype/period, taaki chunk apne aap mein meaningful rahe
            context_prefix = f"Company: {company}\nDocument: {doc_type}\nPeriod: {period}\n\n"
            all_chunks.append({
                "content": context_prefix + chunk,   # embedding ke liye (context ke saath)
                "raw_content": chunk,                 # BM25 aur citation ke liye (bina context)
                "company": company,
                "doc_type": doc_type,
                "period": period,
                "chunk_id": f"{doc_name}_chunk{i}",
            })
    return all_chunks


all_chunks = []
for name, parsed in parsed_by_company.items():
    company_chunks = build_contextual_chunks(parsed, company=name)
    all_chunks.extend(company_chunks)
    print(f"{name} chunks: {len(company_chunks)}")

print(f"Total chunks: {len(all_chunks)}")

## 5. Hybrid retrieval — dense (FAISS) + BM25 + RRF fusion

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

texts_to_embed = [c["content"] for c in all_chunks]  # context-prefixed text
embeddings = embed_model.encode(texts_to_embed, show_progress_bar=True, normalize_embeddings=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # cosine similarity (normalized embeddings pe inner product)
index.add(embeddings.astype("float32"))
print(f"Index vectors: {index.ntotal}")

In [ ]:
from rank_bm25 import BM25Okapi

# raw_content use karo (bina context prefix) — warna repeated "Company: X" score skew karega
tokenized_corpus = [c["raw_content"].lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def dense_search(query, top_k=10):
    query_embedding = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(query_embedding, top_k)
    return list(indices[0])

def bm25_search(query, top_k=10):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return list(top_idx)

def reciprocal_rank_fusion(rankings_list, k=60):
    # RRF: score(doc) = sum over each ranking of 1/(k+rank)
    fused_scores = {}
    for ranking in rankings_list:
        for rank, doc_idx in enumerate(ranking):
            fused_scores[doc_idx] = fused_scores.get(doc_idx, 0) + 1 / (k + rank + 1)
    sorted_docs = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_idx for doc_idx, score in sorted_docs]

def hybrid_search(query, top_k=5):
    dense_results = dense_search(query, top_k=10)
    bm25_results = bm25_search(query, top_k=10)
    fused = reciprocal_rank_fusion([dense_results, bm25_results])
    return fused[:top_k]


def hybrid_search_within(query, candidate_indices, top_k=10):
    """
    hybrid_search() jaisa hi dense+BM25+RRF, but sirf candidate_indices ke
    subset ke andar ranking karta hai — global ranking nahi. Isse pre-filtering
    hoti hai (company/period jaisa metadata filter pehle, ranking baad mein),
    jo post-filtering (global search -> phir filter) se better hai kyunki
    top_k slots kabhi irrelevant companies/periods pe waste nahi hote.
    """
    if not candidate_indices:
        return []
    candidate_indices = list(candidate_indices)

    # dense: candidate subset ke embeddings ke saath cosine similarity
    # (embeddings normalize hue the section 5 mein, isliye dot product = cosine sim)
    query_embedding = embed_model.encode([query], normalize_embeddings=True).astype("float32")[0]
    candidate_embeddings = embeddings[candidate_indices]
    dense_scores = candidate_embeddings @ query_embedding
    dense_ranked = [candidate_indices[i] for i in np.argsort(dense_scores)[::-1]]

    # bm25: bm25.get_scores() poore corpus (all_chunks) ke liye scores deta hai,
    # bina refit kiye hum sirf candidate_indices pe index kar sakte hain
    tokenized_query = query.lower().split()
    bm25_scores_all = bm25.get_scores(tokenized_query)
    bm25_ranked = sorted(candidate_indices, key=lambda idx: bm25_scores_all[idx], reverse=True)

    fused = reciprocal_rank_fusion([dense_ranked, bm25_ranked])
    return fused[:top_k]

## 6. Deterministic financial calculations

In [ ]:
import re
import json

def extract_metric_from_tables(parsed, keywords):
    """
    Saare filings ke tables mein se wo rows dhoondhta hai jinka label
    keyword se match kare (jaise "revenue", "cost of revenue").
    Numbers ko clean karke float mein convert karta hai
    (parentheses = negative, $ aur commas hata deta hai).
    """
    results = {}
    for doc_name, data in parsed.items():
        for table_idx, table in enumerate(data["tables"]):
            for row in table:
                if not row:
                    continue
                label = row[0].lower()
                if any(kw in label for kw in keywords):
                    numbers = []
                    for cell in row[1:]:
                        cleaned = re.sub(r"[^\d\.\-\(\)]", "", cell)
                        if cleaned in ("", "-", "."):
                            continue
                        is_negative = cleaned.startswith("(") and cleaned.endswith(")")
                        cleaned = cleaned.strip("()")
                        try:
                            val = float(cleaned)
                            if is_negative:
                                val = -val
                            numbers.append(val)
                        except ValueError:
                            continue
                    if numbers:
                        results.setdefault(doc_name, []).append({
                            "label": row[0],
                            "values": numbers,
                            "table_idx": table_idx,
                        })
    return results


# ---------- LLM fallback: keyword-match fail hone pe hi chalta hai ----------

learned_keywords = {"revenue": [], "cost": []}
_llm_classification_cache = {}  # (doc_name, metric_type) -> True/False, dobara try na ho


def classify_labels_with_llm(data, company, metric_type):
    """
    Poore document (saari tables) ke row-labels EK HI LLM call mein dikhata hai
    (table-by-table nahi — 10-K mein income statement document ke bahut peeche
    ho sakti hai, isliye pehli N tables tak limit karna miss kar sakta hai).
    """
    candidates = []
    seen = set()
    for table_idx, table in enumerate(data["tables"]):
        for row in table:
            if not row or not row[0].strip():
                continue
            label = row[0].strip()
            key = (table_idx, label.lower())
            if key in seen:
                continue
            seen.add(key)
            sample_values = row[1:3] if len(row) > 1 else []
            candidates.append(f"[table {table_idx}] {label} | {sample_values}")

    if not candidates:
        return None

    candidates = candidates[:500]  # prompt size cap — bahut badi filings ke liye
    context = "\n".join(candidates)

    metric_desc = (
        "total revenue / net sales (top-line income)" if metric_type == "revenue"
        else "cost of revenue / cost of goods sold (direct cost of producing the revenue)"
    )

    prompt = f"""Yahan {company} ke ek SEC filing ke tables ke rows hain (format: [table N] label | sample values):
{context}

Inme se kaunsa row label "{metric_desc}" represent karta hai? Sirf income statement
(consolidated statement of earnings/operations) wali row consider karo, koi
segment-breakdown ya percentage-of-sales row nahi.

Reply ONLY valid JSON, kuch aur text nahi: {{"label": "<exact label as shown>"}}
Agar koi match nahi milta: {{"label": null}}"""

    response = invoke_with_retry(llm, prompt)
    text = extract_text(response).strip()
    text = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()

    try:
        parsed_json = json.loads(text)
        label = parsed_json.get("label")
    except (json.JSONDecodeError, AttributeError):
        return None

    if not label:
        return None
    return label.strip().lower()


def extract_metric_from_tables_with_fallback(parsed, base_keywords, metric_type, company):
    """
    Pehle static keyword-matching try karta hai (fast, free). Agar kisi filing
    ke liye kuch match hi nahi hota, LLM se poore doc ke labels classify
    karwata hai (ek hi call mein) aur naya label 'learned_keywords' mein
    permanently add kar deta hai — taaki agli baar LLM call na lagni pade.
    Result cache hota hai (mila ho ya na mila ho).
    """
    keywords = base_keywords + learned_keywords[metric_type]
    results = extract_metric_from_tables(parsed, keywords)

    for doc_name, data in parsed.items():
        if doc_name in results:
            continue

        cache_key = (doc_name, metric_type)
        if cache_key in _llm_classification_cache:
            continue

        classified_label = classify_labels_with_llm(data, company, metric_type)
        if classified_label and classified_label not in learned_keywords[metric_type]:
            print(f"LLM-learned {metric_type} keyword: \'{classified_label}\' (from {doc_name})")
            learned_keywords[metric_type].append(classified_label)
            keywords = base_keywords + learned_keywords[metric_type]
            results = extract_metric_from_tables(parsed, keywords)
            _llm_classification_cache[cache_key] = True
        else:
            _llm_classification_cache[cache_key] = False
            print(f"LLM could not classify {metric_type} for {doc_name}")

    return results


def calc_growth_rate(old, new):
    if old == 0 or old is None:
        return None
    return round((new - old) / abs(old) * 100, 2)

def calc_cagr(begin_value, end_value, years):
    if begin_value <= 0 or years <= 0:
        return None
    return round(((end_value / begin_value) ** (1 / years) - 1) * 100, 2)

def calc_margin(revenue, cost):
    if revenue == 0 or revenue is None:
        return None
    return round((revenue - cost) / revenue * 100, 2)

## 7. LangGraph agent

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    query: str                  # user ka sawaal
    company: str                 # active entity
    sub_queries: List[str]       # Plan node ke sub-questions
    evidence: List[dict]         # Retrieve node ke chunks
    verification_notes: str      # Verify node ka verdict
    iteration: int               # retry counter
    calculations: dict           # Calculate node ka output
    final_report: str            # Synthesize node ka final answer

In [ ]:
def detect_recency_intent(query):
    """
    Query mein 'recent/latest/current' jaisa recency-signal hai ya nahi.
    Agar hai, to retrieval ko sirf latest filings tak restrict karna chahiye —
    warna embeddings/BM25 purane aur naye period ke chunks ko barabar relevant
    maan lete hain (recency semantic similarity se nahi, metadata se pata chalta hai).
    """
    recency_keywords = ["recent", "recently", "latest", "current", "most recent",
                         "this quarter", "last quarter", "now", "trend"]
    q = query.lower()
    return any(kw in q for kw in recency_keywords)


def detect_doc_type_intent(query):
    """
    Query "quarterly" maang rahi hai ya "annual" — agar clear signal hai to
    sirf usi doc_type tak target restrict karo, warna dono include karo (None).
    Isse "most recent QUARTERLY revenue" jaisi query ke liye latest 10-K
    candidate pool mein aake precision drag down nahi karega.
    """
    q = query.lower()
    if "quarterly" in q or "quarter" in q:
        return "10-Q"
    if "annual" in q or "yearly" in q or "fiscal year" in q:
        return "10-K"
    return None


def get_recent_target_docs(company, doc_type_filter=None):
    """
    Company ki filings mein se har doc_type (10-K, 10-Q) ka sabse recent doc_name
    nikaal ke dict return karta hai, e.g. {"10-K": "10-K_2026-01-25", "10-Q": "10-Q_2026-07-26"}.
    doc_type_filter diya ho to sirf usi type ka latest doc return hota hai
    (query ne "quarterly"/"annual" explicitly maanga ho tab).
    Period format YYYY-MM-DD hai isliye string-sort chronological hi kaam karta hai.
    """
    filings = parsed_by_company.get(company, {})
    doc_types = [doc_type_filter] if doc_type_filter else ["10-K", "10-Q"]
    result = {}
    for doc_type in doc_types:
        docs = sorted([d for d in filings if d.startswith(doc_type)])
        if docs:
            result[doc_type] = docs[-1]
    return result


def contextualize_node(state: AgentState) -> AgentState:
    print("-> Contextualize node running...")
    # keyword-match company naam ya ticker se — abhi simple, sophisticated
    # entity-resolution (NER/embedding-match) baad mein
    query_lower = state["query"].lower()
    detected = None
    for name, ticker in COMPANIES.items():
        if name.lower() in query_lower or ticker.lower() in query_lower:
            detected = name
            break
    state["company"] = detected or next(iter(COMPANIES))  # default: pehli company
    return state


def plan_node(state: AgentState) -> AgentState:
    print("-> Plan node running...")
    prompt = f"""Given this financial research question about {state['company']}, break it into 2-4 specific, focused sub-questions that would help research the answer. Return ONLY the sub-questions, one per line, no numbering, no extra text.

Question: {state['query']}"""
    response = invoke_with_retry(llm, prompt)
    text = extract_text(response)
    state["sub_queries"] = [line.strip() for line in text.split("\n") if line.strip()]
    return state


def retrieve_node(state: AgentState) -> AgentState:
    print("-> Retrieve node running...")
    state["iteration"] += 1  # yahan increment karo (asli node hai, iska return persist hota hai)

    seen_chunk_ids = {e["chunk_id"] for e in state.get("evidence", [])}
    all_evidence = state.get("evidence", [])
    queries_to_search = state["sub_queries"] if state["sub_queries"] else [state["query"]]

    # FIX (recency), v2 — PRE-filter karo, POST-filter nahi:
    # pehle candidate pool banao (company, +period agar recency-intent hai),
    # phir usi pool ke andar ranking karo. Post-filter (pehle global top-15
    # nikalo, phir company/period filter karo) mein top-15 slots zyada waste
    # ho jaate the irrelevant companies pe, evidence starve ho raha tha.
    recency = detect_recency_intent(state["query"])
    doc_type_intent = detect_doc_type_intent(state["query"]) if recency else None
    target_docs = get_recent_target_docs(state["company"], doc_type_filter=doc_type_intent) if recency else None
    target_prefixes = tuple(target_docs.values()) if target_docs else None

    candidate_indices = [
        i for i, c in enumerate(all_chunks)
        if c["company"] == state["company"]
        and (not target_prefixes or c["chunk_id"].startswith(target_prefixes))
    ]
    # Fallback: company ke paas target period ki filing hi na ho (rare), to
    # sirf company-filter tak wapas aa jao — kabhi khaali candidate pool na ho
    if recency and not candidate_indices:
        candidate_indices = [i for i, c in enumerate(all_chunks) if c["company"] == state["company"]]

    for sq in queries_to_search:
        for idx in hybrid_search_within(sq, candidate_indices, top_k=15):
            chunk = all_chunks[idx]
            if chunk["chunk_id"] not in seen_chunk_ids:
                all_evidence.append(chunk)
                seen_chunk_ids.add(chunk["chunk_id"])

    state["evidence"] = all_evidence
    return state


def verify_node(state: AgentState) -> AgentState:
    print("-> Verify node running...")
    evidence_text = "\n\n".join(
        f"[{e['chunk_id']}] {e['raw_content'][:300]}" for e in state["evidence"][:8]
    )
    prompt = f"""Question: {state['query']}

Evidence collected so far:
{evidence_text}

Is this evidence sufficient to answer the question well? Reply with exactly one word first (SUFFICIENT or INSUFFICIENT), then a brief one-line reason."""
    response = invoke_with_retry(llm, prompt)
    state["verification_notes"] = extract_text(response)
    return state


def should_continue_research(state: AgentState) -> str:
    # sirf read karta hai, mutate NAHI karta (routing function ka mutation persist nahi hota)
    if state["iteration"] >= 3:
        return "calculate"
    if "INSUFFICIENT" in state["verification_notes"].upper():
        return "retrieve"
    return "calculate"


def calculate_node(state: AgentState) -> AgentState:
    print("-> Calculate node running...")
    # sirf active company ki filings pe calculation karo
    # FIX (multi-company): generic dict lookup, koi company hardcode nahi
    company_filings = parsed_by_company.get(state["company"], {})

    # pehle static keyword-matching try hota hai; agar kisi filing ke liye
    # kuch match nahi hota, LLM fallback us doc ke labels classify karta hai
    # (ek baar; result learned_keywords mein permanently cache ho jaata hai)
    revenue_data = extract_metric_from_tables_with_fallback(
        company_filings, base_keywords=["total revenue", "revenue", "net revenue", "net sales"],
        metric_type="revenue", company=state["company"],
    )
    cost_data = extract_metric_from_tables_with_fallback(
        company_filings, base_keywords=["cost of revenue", "cost of sales", "cost of products sold"],
        metric_type="cost", company=state["company"],
    )

    calculations = {}
    for doc_name in revenue_data:
        if doc_name not in cost_data:
            print(f"No cost-of-revenue match for {doc_name} — check filing's actual label wording")
            continue

        # Same-table match dhoondo, warna nearby tables allow karo (window=2)
        # — kuch filings (especially 10-K) mein ek hi income statement
        # multiple adjacent <table> tags mein split hoti hai
        matched = None
        best_gap = None
        for rev_row in revenue_data[doc_name]:
            for cost_row in cost_data[doc_name]:
                gap = abs(rev_row["table_idx"] - cost_row["table_idx"])
                if gap <= 2 and (best_gap is None or gap < best_gap):
                    matched = (rev_row, cost_row)
                    best_gap = gap

        if not matched:
            print(f"No nearby revenue/cost pair for {doc_name} (checked window=2)")
            continue

        rev_row, cost_row = matched
        if rev_row["values"] and cost_row["values"]:
            revenue = rev_row["values"][0]
            cost = cost_row["values"][0]
            # sanity check: cost, revenue ka 10%-95% ke beech hona chahiye
            # (isse bahar hai matlab galat row match hua, koi ratio/percentage row)
            if revenue > 0 and (0.10 * revenue <= cost <= 0.95 * revenue):
                calculations[doc_name] = {
                    "revenue": revenue,
                    "cost_of_revenue": cost,
                    "gross_margin_pct": calc_margin(revenue, cost),
                }
            else:
                print(f"Skipped {doc_name}: cost={cost}, revenue={revenue}")  # sanity check fail

    # quarterly growth — doc_names YYYY-MM-DD format mein hain, string-sort chronological hi hai
    quarterly_docs = sorted([d for d in calculations if d.startswith("10-Q")])
    if len(quarterly_docs) >= 2:
        oldest, newest = quarterly_docs[0], quarterly_docs[-1]
        growth = calc_growth_rate(calculations[oldest]["revenue"], calculations[newest]["revenue"])
        calculations["_quarterly_revenue_growth"] = {"from": oldest, "to": newest, "growth_pct": growth}

    state["calculations"] = calculations
    return state


def synthesize_node(state: AgentState) -> AgentState:
    print("-> Synthesize node running...")
    evidence_text = "\n\n".join(
        f"[{e['chunk_id']}] {e['raw_content'][:400]}" for e in state["evidence"][:8]
    )

    calc_text = ""
    if state["calculations"]:
        calc_lines = []
        for k, v in state["calculations"].items():
            if not k.startswith("_"):
                calc_lines.append(f"{k}: revenue=${v['revenue']}M, gross_margin={v['gross_margin_pct']}%")
            else:
                calc_lines.append(f"{k}: {v}")
        calc_text = "Deterministically calculated financial metrics:\n" + "\n".join(calc_lines)

    prompt = f"""You are a financial research analyst. Answer the following question using the evidence and calculations provided.

IMPORTANT: You MUST explicitly cite the specific numeric values from "Deterministically calculated financial metrics" section below (exact percentages, dollar figures) in your answer, not just qualitative statements. Cite chunk IDs in brackets like [chunk_id] when referencing text evidence.

Question: {state['query']}

Evidence:
{evidence_text}

{calc_text}

Write a clear, well-organized answer that includes the specific calculated numbers."""
    response = invoke_with_retry(llm, prompt)
    state["final_report"] = extract_text(response)
    return state

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("contextualize", contextualize_node)
graph.add_node("plan", plan_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("verify", verify_node)
graph.add_node("calculate", calculate_node)
graph.add_node("synthesize", synthesize_node)

graph.set_entry_point("contextualize")
graph.add_edge("contextualize", "plan")
graph.add_edge("plan", "retrieve")
graph.add_edge("retrieve", "verify")
graph.add_conditional_edges("verify", should_continue_research, {"retrieve": "retrieve", "calculate": "calculate"})
graph.add_edge("calculate", "synthesize")
graph.add_edge("synthesize", END)

app = graph.compile()
print("Graph compiled")

## 8. End-to-end tests

In [ ]:
def run_query(query):
    result = app.invoke({
        "query": query, "company": "", "sub_queries": [], "evidence": [],
        "verification_notes": "", "iteration": 0,
        "calculations": {}, "final_report": "",
    })
    return result


test_questions = {
    "NVIDIA": "Why did NVIDIA's gross margin change recently?",
    "AMD": "What is AMD's revenue trend recently?",
    "Apple": "What is Apple's revenue trend recently?",
    "Walmart": "How has Walmart's gross margin trended recently?",
    "Johnson & Johnson": "What is Johnson & Johnson's revenue trend recently?",
    "JPMorgan Chase": "What is JPMorgan Chase's revenue trend recently?",
}

results_by_company = {}
for company, question in test_questions.items():
    print(f"### {company} ###")
    result = run_query(question)
    results_by_company[company] = result
    print(f"{company}:\n", result["final_report"])
    print("\n" + "=" * 80 + "\n")

# backward-compat aliases (pehle wali eval cells inhi naamon se refer karti hain)
result_nvda = results_by_company.get("NVIDIA")
result_amd = results_by_company.get("AMD")

## 9. Evaluation

Teen self-contained checks — inko ground-truth ki zaroorat nahi (sirf golden-set accuracy check ke liye thoda manual data chahiye, wo optional hai):

1. **Retrieval contamination** — `retrieve_node` kitni baar galat company ka chunk le aata hai
2. **Calculation coverage** — `calculate_node` kitne filings ke liye valid revenue/cost nikal paata hai (aur kaunse skip ho rahe hain)
3. **Citation groundedness** — `synthesize_node` ke final answer mein jo `[chunk_id]` cite hue, kya wo actually retrieved evidence mein the

In [ ]:
def eval_retrieval_contamination(test_queries):
    """
    test_queries: dict {"NVIDIA": [q1, q2, ...], "AMD": [q1, q2, ...], ...}
    Har query ko us company ke saath retrieve_node se chalata hai aur check karta hai
    ki evidence mein sirf usi company ke chunks aaye ya nahi.
    """
    rows = []
    for company, queries in test_queries.items():
        for q in queries:
            state = {
                "query": q, "company": company, "sub_queries": [], "evidence": [],
                "verification_notes": "", "iteration": 0, "calculations": {}, "final_report": "",
            }
            state = retrieve_node(state)
            contaminated = [e["chunk_id"] for e in state["evidence"] if e["company"] != company]
            rows.append({
                "company": company,
                "query": q,
                "n_evidence": len(state["evidence"]),
                "n_contaminated": len(contaminated),
                "contaminated_ids": contaminated,
            })

    total_evidence = sum(r["n_evidence"] for r in rows)
    total_contaminated = sum(r["n_contaminated"] for r in rows)
    contamination_rate = round(100 * total_contaminated / total_evidence, 2) if total_evidence else None

    print(f"Total evidence chunks checked: {total_evidence}")
    print(f"Contaminated chunks: {total_contaminated}")
    print(f"Contamination rate: {contamination_rate}%")
    for r in rows:
        flag = "  <-- CONTAMINATED" if r["n_contaminated"] else ""
        company = r["company"]
        query = r["query"]
        n_ev = r["n_evidence"]
        n_bad = r["n_contaminated"]
        print(f'[{company}] "{query}" -> {n_ev} chunks, {n_bad} bad{flag}')

    return {"rows": rows, "contamination_rate_pct": contamination_rate}


# test queries — har company ke liye 2 query: ek company-naam ke saath, ek generic
test_queries = {
    "NVIDIA": ["What drove NVIDIA's gross margin change recently?", "How much did NVIDIA spend on R&D?"],
    "AMD": ["What is AMD's revenue trend recently?", "How much did AMD spend on R&D?"],
    "Apple": ["What is Apple's revenue trend recently?", "How much did Apple spend on R&D?"],
    "Walmart": ["What is Walmart's revenue trend recently?", "How has Walmart's gross margin changed?"],
    "Johnson & Johnson": ["What is Johnson & Johnson's revenue trend recently?", "How has JNJ's gross margin changed?"],
    "JPMorgan Chase": ["What is JPMorgan Chase's revenue trend recently?", "How has JPMorgan's revenue changed?"],
}

contamination_results = eval_retrieval_contamination(test_queries)

In [ ]:
def eval_calculation_coverage():
    """
    calculate_node dobara call NAHI karta — results_by_company (section 8) mein
    calculations already computed hain, isliye wahi reuse karte hain. Pehle ye
    function har company ke liye calculate_node fresh se chalata tha, jo
    poori tarah redundant tha (same result, extra compute time).
    """
    summary = {}
    for company, filings in parsed_by_company.items():
        print(f"--- {company} ---")
        calcs = results_by_company[company]["calculations"]
        calculated_docs = [k for k in calcs if not k.startswith("_")]
        total_docs = len(filings)
        coverage_pct = round(100 * len(calculated_docs) / total_docs, 1) if total_docs else None

        summary[company] = {
            "total_filings": total_docs,
            "calculated": len(calculated_docs),
            "coverage_pct": coverage_pct,
            "calculated_docs": calculated_docs,
            "missing_docs": [d for d in filings if d not in calculated_docs],
        }
        print(f"Coverage: {len(calculated_docs)}/{total_docs} filings ({coverage_pct}%)")
        missing = summary[company]["missing_docs"]
        print(f"Missing: {missing}")
        print()

    return summary


coverage_results = eval_calculation_coverage()

In [ ]:
def eval_citation_groundedness(result):
    """
    run_query() ka output result dict leta hai. Final report mein jo bhi
    [chunk_id] cite hue hain, unhe evidence list ke actual chunk_ids se
    match karta hai — koi hallucinated citation toh nahi.
    """
    cited = set(re.findall(r"\[([\w\-]+_chunk\d+)\]", result["final_report"]))
    evidence_ids = {e["chunk_id"] for e in result["evidence"]}

    grounded = cited & evidence_ids
    ungrounded = cited - evidence_ids
    groundedness_pct = round(100 * len(grounded) / len(cited), 1) if cited else None

    print(f"Citations found: {len(cited)}")
    print(f"Grounded (in evidence): {len(grounded)}")
    print(f"Ungrounded (hallucinated chunk_id): {len(ungrounded)}")
    if ungrounded:
        print(f"  -> {ungrounded}")
    print(f"Groundedness: {groundedness_pct}%")

    return {
        "n_cited": len(cited),
        "n_grounded": len(grounded),
        "ungrounded_ids": list(ungrounded),
        "groundedness_pct": groundedness_pct,
    }


# results_by_company Cell 26 (run_query) se already state mein hai — sab 6 companies ke liye
groundedness_results = {}
for company, result in results_by_company.items():
    print(f"--- {company} report groundedness ---")
    groundedness_results[company] = eval_citation_groundedness(result)
    print()

# backward-compat aliases
groundedness_nvda = groundedness_results["NVIDIA"]
groundedness_amd = groundedness_results["AMD"]

### Eval summary

In [ ]:
print("=" * 50)
print("EVAL SUMMARY")
print("=" * 50)
print(f"Retrieval contamination rate: {contamination_results['contamination_rate_pct']}%  (target: 0%)")
print()
for company, s in coverage_results.items():
    print(f"{company} calculation coverage: {s['coverage_pct']}%  ({s['calculated']}/{s['total_filings']} filings)")
print()
for company, g in groundedness_results.items():
    print(f"{company} citation groundedness: {g['groundedness_pct']}%")

## 10. Baseline comparison — naive RAG vs agentic system

Ek minimal "naive RAG" banate hain jisme wo saari cheezein missing hain jo humne is conversation mein add ki:
- **No company filter** — pura `all_chunks` search hota hai, company ignore karke (Bug 1 wala original behavior)
- **No deterministic calculation** — LLM ko seedha raw text se numbers nikaalne ko bola jaata hai
- **Single-pass retrieval** — koi verify-loop, koi sub-query expansion nahi, ek hi `hybrid_search` call

Same 6 test questions dono systems pe chalate hain, aur measure karte hain: **contamination rate** aur **citation groundedness**. Ye numbers directly justify karte hain ki agentic design (company-filter + verify-loop + deterministic-calc) ne kya improve kiya.

In [ ]:
def naive_run_query(query, company):
    """
    Minimal RAG: ek hi retrieval pass, company-filter nahi, deterministic
    calculation nahi — sab kuch LLM ke upar chhod diya. Compare karne ke liye
    baseline ke roop mein use hoga.
    """
    top_indices = hybrid_search(query, top_k=10)
    evidence = [all_chunks[idx] for idx in top_indices]

    evidence_text = "\n\n".join(
        f"[{e['chunk_id']}] {e['raw_content'][:400]}" for e in evidence
    )
    prompt = f"""Question: {query}

Evidence:
{evidence_text}

Answer the question using the evidence above. Cite chunk_ids in [brackets] where relevant."""

    response = invoke_with_retry(llm, prompt)
    final_report = extract_text(response)

    return {"final_report": final_report, "evidence": evidence, "company": company}


def eval_contamination_and_groundedness(result, expected_company):
    evidence_ids = {e["chunk_id"] for e in result["evidence"]}
    contaminated = [e["chunk_id"] for e in result["evidence"] if e["company"] != expected_company]

    cited = set(re.findall(r"\[([\w\-]+_chunk\d+)\]", result["final_report"]))
    grounded = cited & evidence_ids
    groundedness_pct = round(100 * len(grounded) / len(cited), 1) if cited else None

    return {
        "n_evidence": len(result["evidence"]),
        "n_contaminated": len(contaminated),
        "n_cited": len(cited),
        "groundedness_pct": groundedness_pct,
    }

In [ ]:
comparison_rows = []

for company, question in test_questions.items():
    print(f"=== {company} ===")

    # Naive baseline
    naive_result = naive_run_query(question, company)
    naive_metrics = eval_contamination_and_groundedness(naive_result, company)

    # Agentic system (already computed results_by_company mein hai agar run_query chal chuka hai)
    agentic_result = results_by_company.get(company) or run_query(question)
    agentic_metrics = eval_contamination_and_groundedness(agentic_result, company)

    comparison_rows.append({
        "company": company,
        "naive_contamination": naive_metrics["n_contaminated"],
        "agentic_contamination": agentic_metrics["n_contaminated"],
        "naive_groundedness_pct": naive_metrics["groundedness_pct"],
        "agentic_groundedness_pct": agentic_metrics["groundedness_pct"],
        "naive_has_calc_section": "calculat" in naive_result["final_report"].lower(),
        "agentic_has_calc_section": bool(agentic_result.get("calculations")),
    })

    print(f"  Naive:   contamination={naive_metrics['n_contaminated']}, groundedness={naive_metrics['groundedness_pct']}%")
    print(f"  Agentic: contamination={agentic_metrics['n_contaminated']}, groundedness={agentic_metrics['groundedness_pct']}%")
    print()

In [ ]:
print("=" * 70)
print(f"{'Company':<20} {'Naive contam.':<15} {'Agentic contam.':<17} {'Naive ground%':<15} {'Agentic ground%'}")
print("=" * 70)
for r in comparison_rows:
    print(f"{r['company']:<20} {r['naive_contamination']:<15} {r['agentic_contamination']:<17} "
          f"{str(r['naive_groundedness_pct']):<15} {r['agentic_groundedness_pct']}")

total_naive_contam = sum(r["naive_contamination"] for r in comparison_rows)
total_agentic_contam = sum(r["agentic_contamination"] for r in comparison_rows)
n_naive_calc = sum(r["naive_has_calc_section"] for r in comparison_rows)
n_agentic_calc = sum(r["agentic_has_calc_section"] for r in comparison_rows)

print()
print(f"Total contamination — Naive: {total_naive_contam}, Agentic: {total_agentic_contam}")
print(f"Queries with deterministic-calc numbers — Naive: {n_naive_calc}/{len(comparison_rows)}, Agentic: {n_agentic_calc}/{len(comparison_rows)}")

## 11. Retrieval quality — Precision@k / Recall@k

Har company ke **sabse recent target filing** (query mein "quarterly" bola ho to 10-Q, "annual" bola ho to 10-K) ko target bana ke query chalate hain, aur seedha production `retrieve_node` ko call karte hain (koi alag eval-only search logic nahi) — taaki number wahi ho jo deployed system dega.

- **Precision@k** — retrieved chunks mein se kitne % target filing ke hain
- **Recall@k** — target filing ke total chunks mein se kitne % top-k mein aa paaye (naturally `k / n_avail` se capped hota hai jab filing bade ho)

In [ ]:
def eval_precision_recall_at_k(k=10):
    """
    hybrid_search_scoped ke bajaye seedha retrieve_node call karta hai —
    isse recency-fix ka asli effect measure hota hai (jo production mein
    actually chalega), koi eval-only shortcut nahi.
    """
    rows = []
    for company, filings in parsed_by_company.items():
        quarterly_docs = sorted([d for d in filings if d.startswith("10-Q")])
        if not quarterly_docs:
            continue
        target_doc = quarterly_docs[-1]

        query = f"What is {company}'s most recent quarterly revenue and gross margin?"
        state = {
            "query": query, "company": company, "sub_queries": [], "evidence": [],
            "verification_notes": "", "iteration": 0, "calculations": {}, "final_report": "",
        }
        state = retrieve_node(state)  # production node, direct call
        retrieved = state["evidence"][:k]

        relevant_retrieved = [c for c in retrieved if c["chunk_id"].startswith(target_doc)]
        precision = round(len(relevant_retrieved) / len(retrieved), 3) if retrieved else 0.0

        total_relevant_chunks = [
            c for c in all_chunks if c["company"] == company and c["chunk_id"].startswith(target_doc)
        ]
        recall = round(len(relevant_retrieved) / len(total_relevant_chunks), 3) if total_relevant_chunks else None

        rows.append({
            "company": company,
            "target_doc": target_doc,
            "n_retrieved": len(retrieved),
            "n_relevant_available": len(total_relevant_chunks),
            "precision_at_k": precision,
            "recall_at_k": recall,
        })

    print(f"{'Company':<20} {'Target doc':<20} {'n_ret':<7} {'n_avail':<9} {'Precision@k':<13} {'Recall@k'}")
    print("-" * 85)
    for r in rows:
        print(f"{r['company']:<20} {r['target_doc']:<20} {r['n_retrieved']:<7} {r['n_relevant_available']:<9} "
              f"{r['precision_at_k']:<13} {r['recall_at_k']}")

    avg_precision = round(sum(r["precision_at_k"] for r in rows) / len(rows), 3) if rows else None
    valid_recalls = [r["recall_at_k"] for r in rows if r["recall_at_k"] is not None]
    avg_recall = round(sum(valid_recalls) / len(valid_recalls), 3) if valid_recalls else None
    print(f"\nAvg Precision@{k} (recency-fix, production retrieve_node): {avg_precision}"
          f"   Avg Recall@{k}: {avg_recall}")

    return rows


precision_recall_results_production = eval_precision_recall_at_k(k=10)